# Fig 8: TBT Heat Map Difference (300W vs 200W)

**Requires sudo:** `sudo nvidia-smi -i 1,2,3 -pl 200` to set the 200W power cap before collecting.  
**Output:** `paper/figures/section3/output/<MODEL_SHORT>/decode_grid_heatmap_diff.{pdf,png}`

> ⚠️ `plot_decode_grid_diff.py` reads `decode_grid_cell_summary.csv` from `paper/figures/section3/output/<MODEL_SHORT>/{300W,200W}/` (pre-reorganization paths, not yet updated to `model_outputs/`). The collect cell checks for the 300W summary at that legacy location. Copy `decode_grid_cell_summary.csv` from `model_outputs/<MODEL_SHORT>/paper/section3/fig4/` into `paper/figures/section3/output/<MODEL_SHORT>/300W/` if needed.

### Call order
1. Ensure `paper/figures/section3/output/<MODEL_SHORT>/300W/decode_grid_cell_summary.csv` exists (see Fig 4 notebook; copy from `model_outputs/<MODEL_SHORT>/paper/section3/fig4/` if needed)
2. Collect 200W decode-grid data: `sudo nvidia-smi -i 1,2,3 -pl 200`, then `OUT_DIR=<repo>/paper/figures/section3/output/<MODEL_SHORT>/200W/decode_grid_data bash profiling/launch_decode_grid.sh`, then `python scripts/plot_decode_grid.py` (writes `decode_grid_cell_summary.csv` to the legacy 200W dir)
3. `scripts/plot_decode_grid_diff.py`

In [ ]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)


def run(cmd):
    buf = []
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return "".join(buf)


In [ ]:
# Steps 1-2: collect decode-grid data at both power levels. Skip if summaries already exist.
# Note: 300W data is also required by Fig 4 — see that notebook to collect it.
# For 200W: set GPU power cap before running (requires sudo):
#   sudo nvidia-smi -i <GPU_ID> -pl 200
# Then run launch_decode_grid.sh with OUT_DIR pointing to output/<MODEL_SHORT>/200W/decode_grid_data/,
# and run plot_decode_grid.py (it picks up DATA/OUT from MODEL_SHORT automatically).
import sys; sys.path.insert(0, str(REPO_ROOT / "config"))
from config import MODEL_SHORT

summary_300w = REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "300W" / "decode_grid_cell_summary.csv"
if not summary_300w.exists():
    run(["bash", str(REPO_ROOT / "profiling/launch_decode_grid.sh")])
    run(["python", str(REPO_ROOT / "paper/figures/section3/scripts/plot_decode_grid.py")])

In [ ]:
# Step 3: plot diff (requires both output/300W/ and output/200W/ summaries)
%matplotlib inline
%run ../scripts/plot_decode_grid_diff.py

In [ ]:
from IPython.display import Image
import sys; sys.path.insert(0, str(REPO_ROOT / "config"))
from config import MODEL_SHORT

Image(str(REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "decode_grid_heatmap_diff.png"))